In [0]:
dbutils.widgets.text("catalog", "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_gold", "gold")

catalog = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

silver_customers_table = f"{catalog}.{schema_silver}.silver_customers"
silver_transactions_table = f"{catalog}.{schema_silver}.silver_transactions"
silver_accounts_table = f"{catalog}.{schema_silver}.silver_accounts"
silver_credit_table = f"{catalog}.{schema_silver}.silver_credit"
silver_branches_table = f"{catalog}.{schema_silver}.silver_branches"
gold_table_full = f"{catalog}.{schema_gold}.gold_customer_360"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Account Aggregation
account_agg = (
    spark.table(silver_accounts_table)
    .groupBy("customer_id")
    .agg(
        F.count("account_id").alias("total_accounts"),
        F.sum("balance").alias("total_balance")
    )
)

# Transaction Aggregation
txn_agg = (
    spark.table(silver_transactions_table).alias("t")
    .join(
        spark.table("banking.silver.silver_accounts").alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "inner"
    )
    .groupBy("a.customer_id")
    .agg(
        F.count("t.txn_id").alias("total_transactions"),
        F.sum("t.amount").alias("total_transaction_amount")
    )
    .withColumnRenamed("customer_id", "customer_id")
)

# Credit Latest
credit = spark.table(silver_credit_table)
window_spec = Window.partitionBy("customer_id").orderBy(F.col("bureau_pull_date").desc())
credit_latest = (
    credit.withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Customer, Branch tables
customers = spark.table(silver_customers_table )
branches = spark.table(silver_branches_table)

# Final Join
customer_360 = (
    customers
    .join(account_agg, "customer_id", "left")
    .join(txn_agg, "customer_id", "left")
    .join(branches, "branch_code", "left")
    .join(credit_latest, "customer_id", "left")
    .withColumn("customer_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("total_accounts", F.coalesce(F.col("total_accounts"), F.lit(0)))
    .withColumn("total_balance", F.coalesce(F.col("total_balance"), F.lit(0)))
    .withColumn("total_transactions", F.coalesce(F.col("total_transactions"), F.lit(0)))
    .withColumn("total_transaction_amount", F.coalesce(F.col("total_transaction_amount"), F.lit(0)))
    .withColumn(
        "customer_segment",
        F.when(F.col("total_balance") >= 500000, "HIGH_VALUE")
         .when(F.col("total_balance") >= 100000, "MEDIUM_VALUE")
         .otherwise("LOW_VALUE")
    )
    .select(
        "customer_id",
        "customer_name",
        "branch_name",
        "total_accounts",
        "total_balance",
        "total_transactions",
        "total_transaction_amount",
        "credit_score",
        "risk_grade",
        "external_active_loans",
        "external_overdue_amount",
        "customer_segment"
    )
)

customer_360.write.format("delta").mode("overwrite").saveAsTable(gold_table_full)